Test Script

Gravação dos 6 gestos (10s cada, rotulado)

In [1]:
import os
import time
import cv2
import mediapipe as mp
from typing import Optional

# ================= CONFIG =================

GESTURES = {
    "1": "open_palm",
    "2": "closed_hand",
    "3": "v_shape",
    "4": "hand_right",
    "5": "hand_left",
    "6": "thumbs_up",
}

CLIPS_PER_GESTURE = 6
CLIP_SECONDS = 10
TARGET_FPS = 15
FRAME_SIZE = (640, 480)

DATA_DIR = os.getcwd()

# ================= HELPERS =================

def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)

def gesture_folder(subject_id: str, gesture: str) -> str:
    return os.path.join(DATA_DIR, f"subject_{subject_id}", gesture)

def count_clips(folder: str) -> int:
    if not os.path.isdir(folder):
        return 0
    return len([f for f in os.listdir(folder) if f.endswith(".mp4")])

def next_clip_path(subject_id: str, gesture: str) -> str:
    folder = gesture_folder(subject_id, gesture)
    ensure_dir(folder)
    n = count_clips(folder) + 1
    return os.path.join(folder, f"clip_{n:02d}.mp4")

def open_camera(index=0):
    if os.name == "nt":
        cap = cv2.VideoCapture(index, cv2.CAP_DSHOW)
    else:
        cap = cv2.VideoCapture(index, cv2.CAP_V4L2)

    cap.set(cv2.CAP_PROP_FRAME_WIDTH, FRAME_SIZE[0])
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, FRAME_SIZE[1])
    cap.set(cv2.CAP_PROP_FPS, TARGET_FPS)
    return cap

# ================= MAIN =================

def draw_rounded_rect(img, x, y, w, h, r, color, alpha=0.6):
    """Draw a rounded rectangle with transparency."""
    overlay = img.copy()

    # body rectangles
    cv2.rectangle(overlay, (x + r, y), (x + w - r, y + h), color, -1)
    cv2.rectangle(overlay, (x, y + r), (x + w, y + h - r), color, -1)

    # corner circles
    cv2.circle(overlay, (x + r, y + r), r, color, -1)
    cv2.circle(overlay, (x + w - r, y + r), r, color, -1)
    cv2.circle(overlay, (x + r, y + h - r), r, color, -1)
    cv2.circle(overlay, (x + w - r, y + h - r), r, color, -1)

    # blend
    cv2.addWeighted(overlay, alpha, img, 1 - alpha, 0, img)

def draw_label(img, text, x, y, scale=0.55, thickness=2):
    cv2.putText(img, text, (x, y), cv2.FONT_HERSHEY_SIMPLEX, scale, (255, 255, 255), thickness, cv2.LINE_AA)

def draw_progress_bar(img, x, y, w, h, progress):
    """progress in [0,1]"""
    progress = max(0.0, min(1.0, progress))
    # background
    draw_rounded_rect(img, x, y, w, h, r=8, color=(30, 30, 30), alpha=0.55)
    # fill
    fill_w = int((w - 6) * progress)
    if fill_w > 0:
        # solid fill (no alpha needed)
        cv2.rectangle(img, (x + 3, y + 3), (x + 3 + fill_w, y + h - 3), (0, 200, 0), -1)
        cv2.rectangle(img, (x + 3, y + 3), (x + w - 3, y + h - 3), (255, 255, 255), 1)

def main():
    subject_id = input("subject_id (ex: 01): ").strip() or "01"

    cap = open_camera(0)
    if not cap.isOpened():
        raise SystemExit("Camera not available.")

    mp_hands = mp.solutions.hands
    mp_draw = mp.solutions.drawing_utils

    hands = mp_hands.Hands(
        static_image_mode=False,
        max_num_hands=1,
        model_complexity=1,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5,
    )

    current_key = "1"
    current_gesture = GESTURES[current_key]
 
    show_landmarks = True
    recording = False
    paused = False
    writer = None
    start_time = 0.0
    out_path = None
    last_write_time = 0.0
    frame_interval = 1.0 / TARGET_FPS 
    written_frames = 0


    fourcc = cv2.VideoWriter_fourcc(*"mp4v")

    print("\n🎮 Controls:")
    print("  1-7  select gesture")
    print("  r    record")
    print("  p    pause/resume")
    print("  x    cancel recording")
    print("  q    quit\n")
    print("  h    toggle landmarks preview (on/off)")

    while True:
        ok, frame = cap.read()
        if not ok:
            continue

        frame = cv2.resize(frame, FRAME_SIZE)

        # ===== RAW FRAME (for saving) =====
        frame_raw = frame.copy()

        # ===== UI FRAME =====
        frame_ui = frame.copy()

        # ===== MediaPipe (preview only, optional) =====
        hand_detected = False
        if show_landmarks:
            res = hands.process(cv2.cvtColor(frame_ui, cv2.COLOR_BGR2RGB))
            if res.multi_hand_landmarks:
                hand_detected = True
                for h in res.multi_hand_landmarks:
                    mp_draw.draw_landmarks(frame_ui, h, mp_hands.HAND_CONNECTIONS)
        else:
            # lightweight detection flag off
            hand_detected = False

        # ===== HUD (pretty) =====
        done = count_clips(gesture_folder(subject_id, current_gesture))
        status = "REC" if recording else "IDLE"
        if paused:
            status = "PAUSED"

        # top-left card
        draw_rounded_rect(frame_ui, x=12, y=12, w=360, h=98, r=16, color=(20, 20, 20), alpha=0.55)
        draw_label(frame_ui, f"Subject {subject_id}   Gesture Selection 1 to 6", 26, 40, scale=0.55)
        draw_label(frame_ui, f"{current_key} = {current_gesture}  Clips: {done}/{CLIPS_PER_GESTURE}", 26, 66, scale=0.55)

        lm_txt = "ON" if show_landmarks else "OFF"
        hand_txt = "YES" if hand_detected else "NO"
        draw_label(frame_ui, f"Landmarks: {lm_txt}   |  Status: {status}", 26, 92, scale=0.55)

        # right-side mini legend card (keys)
        draw_rounded_rect(frame_ui, x=420, y=12, w=208, h=98, r=16, color=(20, 20, 20), alpha=0.40)
        draw_label(frame_ui, "Keys: r:rec ", 434, 40, scale=0.55)
        draw_label(frame_ui, "x:cancel h:landmarks", 434, 66, scale=0.50)
        draw_label(frame_ui, "q/esc:quit", 434, 90, scale=0.50)

        # ===== Recording overlay =====
        if recording:
            # time/progress
            elapsed = time.time() - start_time
            progress = elapsed / CLIP_SECONDS
            remaining = max(0.0, CLIP_SECONDS - elapsed)

            draw_progress_bar(frame_ui, x=12, y=120, w=616, h=22, progress=progress)
            draw_rounded_rect(frame_ui, x=12, y=150, w=260, h=46, r=14, color=(20, 20, 20), alpha=0.45)
            draw_label(frame_ui, f"Remaining: {remaining:0.1f}s", 26, 180, scale=0.60)

            if not paused and writer:
                now = time.time()
                if now - last_write_time >= frame_interval:
                    writer.write(frame_raw)
                    last_write_time += frame_interval  
                    written_frames += 1

            if not paused and elapsed >= CLIP_SECONDS:
                writer.release()
                writer = None
                recording = False
                print(f"✅ Saved {out_path}")
        # ===== Show UI Frame =====
        cv2.imshow("Gesture Recorder", frame_ui)
        key = cv2.waitKey(1) & 0xFF
        
        # ===== KEYS =====
        if key in (ord("q"), 27):
            break

        if chr(key) in GESTURES:
            current_key = chr(key)
            current_gesture = GESTURES[current_key]
            print("➡ Gesture:", current_gesture)

        if key == ord("r") and not recording:
            out_path = next_clip_path(subject_id, current_gesture)
            writer = cv2.VideoWriter(out_path, fourcc, TARGET_FPS, FRAME_SIZE)
            start_time = time.time()
            last_write_time = start_time
            written_frames = 0
            recording = True
            paused = False
            print(f"⏺ Recording {out_path}")

        if key == ord("p") and recording:
            paused = not paused
            print("⏸ Paused" if paused else "▶ Resumed")

        if key == ord("x") and recording:
            writer.release()
            os.remove(out_path)
            writer = None
            recording = False
            paused = False
            print("❌ Recording cancelled")
        if key == ord("h"):
            show_landmarks = not show_landmarks
            print("👁 Landmarks:", "ON" if show_landmarks else "OFF")


    if writer:
        writer.release()
    hands.close()
    cap.release()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    main()



🎮 Controls:
  1-7  select gesture
  r    record
  p    pause/resume
  x    cancel recording
  q    quit

  h    toggle landmarks preview (on/off)
⏺ Recording c:\Users\jessb\Music\IPT\Aprendizagem computacional\P2\subject_je\open_palm\clip_01.mp4
✅ Saved c:\Users\jessb\Music\IPT\Aprendizagem computacional\P2\subject_je\open_palm\clip_01.mp4
